# LOKLOK v8 ? Joint-Sweep Run (unique output)

Downloads the latest bundle, uses the cached BGE-M3 embeddings and the 185-segment self-report transcript, runs the calibrated in-silico experiment with the 256-point joint mechanism + state-weight sweep, and downloads a uniquely named zip: **loklok_v8_joint_sweep_results.zip**.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
!wget -q -O /content/colab_bundle_v8.zip https://cycles-bless-equal-printers.trycloudflare.com/colab_bundle_v8.zip
!cd /content && rm -rf loklok_v8 && unzip -q -o colab_bundle_v8.zip -d /content
from pathlib import Path
ROOT=Path('/content/loklok_v8')
print(ROOT, ROOT.exists())
print('selfreport:', (ROOT/'03_data/qualitative/selfreport/selfreport_transcript.json').exists())
print('cache:', (ROOT/'03_data/model_outputs/embeddings/embeddings_0f6c66e54fb6ca2c.npy').exists())


In [ ]:
import sys
from pathlib import Path
ROOT=Path('/content/loklok_v8')
sys.path.insert(0,str(ROOT/'02_code/pipeline'))
import loklok_pipeline_v8 as v8
docs,summary=v8.build_text_documents(ROOT/'03_data/raw',ROOT/'03_data/qualitative')
cache=v8._embedding_cache_path(ROOT/'03_data/model_outputs/embeddings','BAAI/bge-m3',docs)
print(summary)
print(cache.name,cache.exists())
assert cache.exists()


In [ ]:
!cd /content/loklok_v8 && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python 02_code/pipeline/loklok_pipeline_v8.py --root /content/loklok_v8 --n-personas 500 --sessions 18 --seed 20261105 --embedding-model BAAI/bge-m3 --device cuda


In [ ]:
!cd /content/loklok_v8 && python 02_code/pipeline/generate_figures.py --root /content/loklok_v8


In [ ]:
import json
from pathlib import Path
ROOT=Path('/content/loklok_v8')
r=json.loads((ROOT/'03_data/model_outputs/pipeline_results.json').read_text(encoding='utf-8'))
print('calibration:',r['calibration'])
print('delta:',r['factorial']['delta_sr_eval']['mean'],r['factorial']['delta_sr_comp']['mean'],r['factorial']['delta_engagement']['mean'])
print('main:',{k:v['mean'] for k,v in r['factorial']['main_effects'].items()})
print('bayes:',{k:r['bayesian'][k] for k in ['tau_mean','tau_hdi_low','tau_hdi_high','ess_tau']})
print('joint:',{k:r['sensitivity']['joint_sweep'].get(k) for k in ['n_draws','delta_min','delta_max','delta_mean','positive_pct','top_drivers']})
print('topology:',r['topology'])
print('persona:',r['persona_robustness'])


In [ ]:
!cd /content && rm -f loklok_v8_joint_sweep_results.zip && zip -q -r loklok_v8_joint_sweep_results.zip loklok_v8/03_data/model_outputs loklok_v8/03_data/measured loklok_v8/03_data/qualitative/selfreport loklok_v8/01_thesis/src/generated loklok_v8/01_thesis/src/figures loklok_v8/04_figures
!ls -lh /content/loklok_v8_joint_sweep_results.zip


In [ ]:
from google.colab import files
files.download('/content/loklok_v8_joint_sweep_results.zip')
